# Chain-depth calibration on Colab

Chooses the chain depths `d` and the middle budget for the main grid. The single agent runs on development problems at several depths, with every prompt holding the same number of equations, so only the dependency path changes with `d`. Useful depths keep single-agent accuracy at the middle budget away from 0% and 100%, so the other systems have room to do better or worse.

These problems use the development seed. The main grid must use a different seed, so prompts are never tuned on test problems.

Use **Runtime > Change runtime type > L4 GPU**. Results go to `MyDrive/small-multi-agents/results/calibrate_depth/`, with the generated problems, a `run.log`, and summary CSV files. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [ ]:
DEPTHS = [1, 2, 4, 6]
EQUATIONS = 14          # per prompt; the distractor chain fills EQUATIONS - 2 * d
PAIRS_PER_DEPTH = 10
DEV_SEED = 3001         # the main grid uses a different seed
BUDGETS = [4000, 8000, 16000]
SAMPLING_SEED = 42

assert all(1 <= 2 * d < EQUATIONS for d in DEPTHS), "each prompt needs at least one distractor update"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/small-multi-agents/results/calibrate_depth"
!mkdir -p {OUT}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [ ]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

## Problems

Generation is deterministic, so rerunning this cell rewrites the same file and resuming stays valid.

In [ ]:
PROBLEMS = f"{OUT}/problems.jsonl"
!rm -f {PROBLEMS}
for d in DEPTHS:
    distractor = EQUATIONS - 2 * d
    !python generate_problems.py --pairs {PAIRS_PER_DEPTH} --seed {DEV_SEED} --chain-length {d} --distractor-length {distractor} >> {PROBLEMS}
!wc -l {PROBLEMS}

## Run

The single agent only, at every budget. Everything the runner prints is also appended to `run.log`.

In [ ]:
from notebook_tools import run_logged

run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", OUT, "--systems", "single_agent",
            "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), "--resume"], f"{OUT}/run.log")

## Results

Accuracy by depth, problem kind, and budget. `length_limit` is the share of problems where the model ran out of tokens before answering.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path(OUT).glob("b*/single_agent.jsonl")):
    for line in path.read_text().splitlines():
        r = json.loads(line)
        rows.append({
            "d": r["chain_length"], "kind": r["kind"], "budget": r["budget"], "pair_id": r["pair_id"],
            "status": r["status"], "correct": r["correct"], "output": r["output_tokens"],
            "total": r["total_tokens"],
        })
df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/summary_problems.csv", index=False)

by_depth = df.groupby(["d", "kind", "budget"]).agg(
    accuracy=("correct", "mean"),
    problems=("correct", "size"),
    length_limit=("status", lambda s: (s == "length_limit").mean()),
    mean_output=("output", "mean"),
).round(2)
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth

In [ ]:
# Accuracy grid: rows are depth and kind, columns are budgets.
by_depth["accuracy"].unstack("budget")

## Reading the results

- Pick the middle budget where the single agent usually finishes (`length_limit` low) but is not near 100% at the larger depths.
- Keep depths whose accuracy at that budget sits roughly between 20% and 80% for at least one kind. Drop depths that are saturated or near zero at every budget.
- If even `d = 6` is near 100%, raise `EQUATIONS` and add larger depths. If `d = 1` is already low, the task is too hard at these budgets.

With 10 pairs per depth, each problem is 10 percentage points within a kind, so treat this as a coarse calibration.